# Preliminary Analysis

Reproduces the preliminary study figures using baseline-only simulation data.
Requires `data/preliminary_results.csv` and `data/preliminary_inventory.csv`.

Run from the `bse/` root directory.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

HERE     = os.getcwd()
PRE_CSV  = os.path.join(HERE, 'data', 'preliminary_results.csv')
PRE_INV  = os.path.join(HERE, 'data', 'preliminary_inventory.csv')
SAVE_DIR = os.path.join(HERE, 'figures')
os.makedirs(SAVE_DIR, exist_ok=True)

INIT_CASH = 10000

plt.rcParams.update({
    'text.usetex':       False,
    'font.family':       'serif',
    'font.serif':        ['DejaVu Serif'],
    'axes.labelsize':    14,
    'font.size':         11,
    'legend.fontsize':   11,
    'xtick.labelsize':   12,
    'ytick.labelsize':   12,
    'axes.titlesize':    13,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         False,
})

CONDITIONS  = ['stable', 'shock_up', 'shock_down', 'multi_regime']
COND_LABELS = ['Stable', 'Shock Up', 'Shock Down', 'Multi-Regime']
HIGH_CONDS  = ['shock_up', 'multi_regime']
LOW_CONDS   = ['stable', 'shock_down']

COLORS = {
    'ZIP':  '#c0392b', 'PRSH': '#1a3a5c', 'ZIC': '#27ae60',
    'CUSTOM': '#2980b9', 'SNPR': '#8e44ad', 'SHVR': '#e67e22', 'GVWY': '#95a5a6',
}

In [ ]:
df  = pd.read_csv(PRE_CSV)
inv = pd.read_csv(PRE_INV)
df['profit_norm'] = df['final_profit'] / INIT_CASH
print(f'Preliminary results: {len(df):,} rows | conditions: {sorted(df.condition.unique())} | traders: {sorted(df.ttype.unique())}')

## 1. ZIP: Mean Inventory Over Session

In [ ]:
records = []
for (cond, ts), grp in inv[inv.ttype == 'ZIP'].groupby(['condition', 'timestep']):
    records.append({'condition': cond, 'timestep': ts, 'mean': grp['mean_inv'].mean()})
agg = pd.DataFrame(records).sort_values(['condition', 'timestep'])

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
axes = axes.flatten()

for ax, cond, label in zip(axes, CONDITIONS, COND_LABELS):
    sub = agg[agg.condition == cond]
    ax.plot(sub['timestep'] * 10, sub['mean'], color=COLORS['ZIP'], linewidth=2.0)
    ax.axhline(0, color='black', linewidth=0.8, alpha=0.4)
    ax.set_title(label)
    ax.set_xlabel('Time step')
    ax.set_ylabel('Mean inventory (units)')
    ax.set_xlim(0, 3600)

fig.suptitle('ZIP: Mean Inventory Over Session', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig1_zip_profit_evolution.png'), bbox_inches='tight')
plt.show()

## 2. ZIP: Inventory vs Profit by Condition

In [ ]:
zip_df = df[df.ttype == 'ZIP']
c_means = [
    (c, zip_df[zip_df.condition == c]['final_inventory'].mean(),
        zip_df[zip_df.condition == c]['profit_norm'].mean())
    for c in CONDITIONS
]

dot_colors = ['#546E7A', '#F44336', '#1565C0', '#7B1FA2']

fig, ax = plt.subplots(figsize=(7, 5))

for (c, inv_m, prf_m), col in zip(c_means, dot_colors):
    label = COND_LABELS[CONDITIONS.index(c)]
    ax.scatter(inv_m, prf_m, color=col, s=160, zorder=5,
               edgecolors='white', linewidth=1.2)
    ax.annotate(label, xy=(inv_m, prf_m), xytext=(inv_m + 8, prf_m + 0.015),
                fontsize=9, color=col)

invs = np.array([x[1] for x in c_means])
prfs = np.array([x[2] for x in c_means])
m, b = np.polyfit(invs, prfs, 1)
x_line = np.linspace(invs.min() - 20, invs.max() + 20, 100)
ax.plot(x_line, m * x_line + b, '--', color='grey', linewidth=1.2, alpha=0.6)

r = np.corrcoef(invs, prfs)[0, 1]
ax.axvline(0, color='black', linewidth=0.7, alpha=0.4)
ax.axhline(0, color='black', linewidth=0.7, alpha=0.4)
ax.set_xlabel('Mean final inventory (units)')
ax.set_ylabel(r'$P\,/\,C_0$')
ax.set_title(f'ZIP: Inventory as Profit Driver  (r = {r:.3f})')
plt.tight_layout()
plt.show()

## 3. Profit by Volatility Regime

In [ ]:
TRADERS = ['ZIP', 'PRSH', 'ZIC']
x = np.arange(len(TRADERS))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))

for i, (label, conds, alpha, hatch) in enumerate([
    ('Low volatility',  LOW_CONDS,  0.90, ''),
    ('High volatility', HIGH_CONDS, 0.60, '///'),
]):
    means = [df[(df.ttype == t) & (df.condition.isin(conds))]['profit_norm'].mean() for t in TRADERS]
    sems  = [df[(df.ttype == t) & (df.condition.isin(conds))]['profit_norm'].sem()  for t in TRADERS]
    ax.bar(x + (i - 0.5) * w, means, w,
           color=[COLORS[t] for t in TRADERS],
           alpha=alpha, hatch=hatch, edgecolor='k', linewidth=0.6,
           yerr=sems, capsize=5, error_kw={'linewidth': 1.2},
           label=label)

ax.axhline(0, color='black', linewidth=0.8, alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(TRADERS)
ax.set_ylabel(r'$P\,/\,C_0$')
ax.set_title('Profit by Volatility Regime')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 4. Statistical Table: Profit by Volatility Regime

Welch t-test (unequal variances). Groups: high volatility = Shock Up + Multi-Regime; low volatility = Stable + Shock Down.

In [ ]:
rows = []
for t in TRADERS:
    hi = df[(df.ttype == t) & (df.condition.isin(HIGH_CONDS))]['profit_norm']
    lo = df[(df.ttype == t) & (df.condition.isin(LOW_CONDS))]['profit_norm']
    stat, p = stats.ttest_ind(hi, lo, equal_var=False)
    d = (hi.mean() - lo.mean()) / np.sqrt((hi.std()**2 + lo.std()**2) / 2)
    rows.append({
        'Trader':        t,
        'Low-vol mean':  f'{lo.mean():.4f}',
        'High-vol mean': f'{hi.mean():.4f}',
        't':             f'{stat:+.3f}',
        'p':             f'{p:.2e}',
        "Cohen's d":     f'{d:+.3f}',
    })

display(pd.DataFrame(rows).set_index('Trader'))

## 5. Inventory Over Session by Condition

In [ ]:
inv_focus = inv[inv.ttype.isin(TRADERS)].copy()

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
axes = axes.flatten()

for ax, cond, label in zip(axes, CONDITIONS, COND_LABELS):
    sub = inv_focus[inv_focus.condition == cond]
    agg = sub.groupby(['ttype', 'timestep'])['mean_inv'].mean().reset_index()
    for t in TRADERS:
        ts = agg[agg.ttype == t].sort_values('timestep')
        ax.plot(ts['timestep'] * 10, ts['mean_inv'],
                color=COLORS[t], linewidth=2.0, label=t)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='-', alpha=0.35)
    ax.set_title(label)
    ax.set_ylabel('Mean inventory (units)')
    ax.set_xlabel('Time step')
    ax.legend(frameon=False)

fig.suptitle('Mean Inventory Over Session by Condition', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Statistical Table: Final Inventory by Volatility Regime

Mann-Whitney U test (non-parametric). Same grouping as Table 4.

In [ ]:
rows = []
for t in TRADERS:
    hi = df[(df.ttype == t) & (df.condition.isin(HIGH_CONDS))]['final_inventory']
    lo = df[(df.ttype == t) & (df.condition.isin(LOW_CONDS))]['final_inventory']
    U, p = stats.mannwhitneyu(hi, lo, alternative='two-sided')
    d = (hi.mean() - lo.mean()) / np.sqrt((hi.std()**2 + lo.std()**2) / 2)
    rows.append({
        'Trader':              t,
        'Low-vol mean inv':    f'{lo.mean():+.1f}',
        'High-vol mean inv':   f'{hi.mean():+.1f}',
        'U':                   f'{U:,.0f}',
        'p':                   f'{p:.2e}',
        "Cohen's d":           f'{d:+.3f}',
    })

display(pd.DataFrame(rows).set_index('Trader'))